In [ ]:
# The pre-requisite for this python file is ncat to be installed 
#docker exec -it <jupyter-lab-process-id>
#sudo apt-get install ncat

In [1]:
from pyspark.sql import SparkSession

spark=(SparkSession
.builder 
.appName("read from socket") 
.config("spark.streaming.stopGracefullyOnShutdown",True)
.master("local[*]") 
.getOrCreate()
      )
spark


In [2]:
#Read Input
spark.conf.set("spark.sql.streaming.schemaInference",True)
stream_df=(spark
           .readStream
           .option("cleanSource","archive")
           .option("sourceArchiveDir","archive_dir")
           .option("maxFilesPerTrigger",1)
           .format("json")
           .load("input/")
          )

In [3]:
stream_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [4]:
#explode
from pyspark.sql.functions import explode

df_explode=stream_df.withColumn("data_devices",explode("data.devices"))
df_explode.printSchema()
# df_explode.show(truncate=False)

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- data_devices: struct (nullable = true)
 |    |-- deviceId: string (nullable = true)
 |    |-- measure: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- temperature: long (nullable = true)



In [5]:
#flatten df
from pyspark.sql.functions import col
flatten_df=(df_explode
            .withColumn("device_id",col("data_devices.deviceid"))
            .withColumn("measure",col("data_devices.measure"))
            .withColumn("status",col("data_devices.status"))
            .withColumn("temperature",col("data_devices.temperature"))
            .drop("data")
            .drop("data_devices"))
# df_agg=df_explode.groupBy("word").agg(count(lit (1)).alias("CNT"))
flatten_df.printSchema()
# flatten_df.show()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)



In [33]:
flatten_df.writeStream.format("console").outputMode("append").start().awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/local/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
(flatten_df
 .writeStream
 .format("csv")
 .outputMode("append")
 .option("path","output/data_out1.csv")
 .option("checkpointLocation","checkpoint_dir")
 .start().awaitTermination())